# System Setup

In [1]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
import os

## Prep to Add C4 parameters

# Phase 0: SPC/E validation

## Figuring out Water Model
- Turns out we need to make sure we use the charmm36_2024/spce.xml rather than our own version to avoid it trying to use the baked-in TIP3P/TP3M models for water... because of course.
- That being said, it seems like we may be able to utilize the SPCE values after all since the CHARMM SPCE model doesn't change anything from the Amber one.

In [2]:
"""
Phase 0, step 1: verify the custom force field XML *before* running any MD.

Builds two water molecules at a fixed, arbitrary geometry and compares
OpenMM's energy against a hand-coded Coulomb + Lennard-Jones sum.
If these disagree, the XML has a units error, a wrong charge, a bad
combining rule, or a mis-assigned LJ site.

Run:  python test_forcefield.py my_charmm_spce.xml
"""

import sys
import itertools
import numpy as np
from openmm import app, unit, Vec3
import openmm as mm

# Canonical SPC/E reference values (Berendsen 1987)
REF = dict(
    q_O=-0.8476, q_H=0.4238,          # e
    sigma_O=0.316557,                 # nm
    epsilon_O=0.650194,               # kJ/mol
    r_OH=0.1, angle=109.47,           # nm, degrees
)

ONE_4PI_EPS0 = 138.935456             # kJ*nm/(mol*e^2)


def water_coords(origin, rotate=False):
    """SPC/E geometry: O at origin, H's in the xy-plane."""
    a = np.radians(REF["angle"]) / 2.0
    o = np.array([0.0, 0.0, 0.0])
    h1 = np.array([REF["r_OH"] * np.sin(a), REF["r_OH"] * np.cos(a), 0.0])
    h2 = np.array([-REF["r_OH"] * np.sin(a), REF["r_OH"] * np.cos(a), 0.0])
    pts = [o, h1, h2]
    if rotate:                        # arbitrary rotation, avoids symmetry
        th = 0.7
        R = np.array([[np.cos(th), 0, np.sin(th)],
                      [0, 1, 0],
                      [-np.sin(th), 0, np.cos(th)]])
        pts = [R @ p for p in pts]
    return [p + np.array(origin) for p in pts]


def build_two_waters():
    top = app.Topology()
    positions = []
    for i, origin in enumerate([(0, 0, 0), (0.45, 0.12, 0.05)]):
        ch = top.addChain()
        res = top.addResidue("HOH", ch)
        o = top.addAtom("O", app.Element.getBySymbol("O"), res)
        h1 = top.addAtom("H1", app.Element.getBySymbol("H"), res)
        h2 = top.addAtom("H2", app.Element.getBySymbol("H"), res)
        top.addBond(o, h1)
        top.addBond(o, h2)
        positions.extend(water_coords(origin, rotate=bool(i)))
    return top, np.array(positions)


def analytic_energy(pos):
    """Coulomb + LJ between the two molecules (intramolecular excluded)."""
    q = [REF["q_O"], REF["q_H"], REF["q_H"]] * 2
    s, e = REF["sigma_O"], REF["epsilon_O"]

    total = 0.0
    for i, j in itertools.product(range(3), range(3, 6)):
        r = np.linalg.norm(pos[i] - pos[j])
        total += ONE_4PI_EPS0 * q[i] * q[j] / r
        if i == 0 and j == 3:                      # O-O LJ only
            sr6 = (s / r) ** 6
            total += 4.0 * e * (sr6 ** 2 - sr6)
    return total


def openmm_energy(xml, top, pos):
    ff = app.ForceField(xml)
    system = ff.createSystem(top,
                             nonbondedMethod=app.NoCutoff,
                             constraints=None,
                             rigidWater=False)   # keep geometry as given
    integ = mm.VerletIntegrator(1.0 * unit.femtosecond)
    ctx = mm.Context(system, integ, mm.Platform.getPlatformByName("Reference"))
    ctx.setPositions([Vec3(*p) for p in pos] * unit.nanometer)
    return ctx.getState(getEnergy=True).getPotentialEnergy().value_in_unit(
        unit.kilojoule_per_mole)


def report_parameters(xml, top):
    """Dump what the XML actually assigned - the fastest way to spot typos."""
    ff = app.ForceField(xml)
    system = ff.createSystem(top, nonbondedMethod=app.NoCutoff,
                             constraints=None, rigidWater=False)
    nb = next(f for f in system.getForces() if isinstance(f, mm.NonbondedForce))
    print("\nassigned nonbonded parameters (molecule 1):")
    for i in range(3):
        q, s, e = nb.getParticleParameters(i)
        print(f"  atom {i}: q={q.value_in_unit(unit.elementary_charge):+.5f} e"
              f"  sigma={s.value_in_unit(unit.nanometer):.6f} nm"
              f"  eps={e.value_in_unit(unit.kilojoule_per_mole):.6f} kJ/mol")
    print("\nexpected:")
    print(f"  O      : q={REF['q_O']:+.5f} e  sigma={REF['sigma_O']:.6f} nm"
          f"  eps={REF['epsilon_O']:.6f} kJ/mol")
    print(f"  H      : q={REF['q_H']:+.5f} e  sigma=(any)     eps=0.000000")
    total_q = sum(nb.getParticleParameters(i)[0].value_in_unit(
        unit.elementary_charge) for i in range(3))
    print(f"\n  molecular charge: {total_q:+.2e} e  (must be ~0)")

xml = 'charmm36_2024/spce.xml'
top, pos = build_two_waters()

report_parameters(xml, top)

e_mm = openmm_energy(xml, top, pos)
e_ref = analytic_energy(pos)
print(f"\nOpenMM   : {e_mm:12.6f} kJ/mol")
print(f"analytic : {e_ref:12.6f} kJ/mol")
print(f"diff     : {e_mm - e_ref:12.2e} kJ/mol")

assert abs(e_mm - e_ref) < 1e-4, "XML does not reproduce SPC/E - stop here"
print("\nPASS")

ValueError: Could not locate file "charmm36_2024/spce.xml"

In [6]:
"""
Locate where a custom XML disagrees with reference SPC/E.

Unlike the first test, this one:
  - looks for LJ in CustomNonbondedForce too (CHARMM XMLs put it there)
  - decomposes the energy by force group, so you see WHICH term is wrong
  - probes LJ numerically by scanning two isolated oxygens
  - reports bonded equilibrium values and constraints

Run:  python diagnose_ff.py my_charmm_spce.xml
"""

import sys
import numpy as np
from openmm import app, unit, Vec3
import openmm as mm

REF = dict(q_O=-0.8476, q_H=0.4238, sigma=0.316557, epsilon=0.650194,
           r_OH=0.1, angle=109.47)
ONE_4PI_EPS0 = 138.935456


def build(res_name="HOH", o_name="O"):
    top = app.Topology()
    pos = []
    a = np.radians(REF["angle"]) / 2.0
    geo = [np.array([0.0, 0.0, 0.0]),
           np.array([REF["r_OH"] * np.sin(a), REF["r_OH"] * np.cos(a), 0.0]),
           np.array([-REF["r_OH"] * np.sin(a), REF["r_OH"] * np.cos(a), 0.0])]
    for origin in [(0, 0, 0), (0.45, 0.12, 0.05)]:
        ch = top.addChain()
        r = top.addResidue(res_name, ch)
        o = top.addAtom(o_name, app.Element.getBySymbol("O"), r)
        h1 = top.addAtom("H1", app.Element.getBySymbol("H"), r)
        h2 = top.addAtom("H2", app.Element.getBySymbol("H"), r)
        top.addBond(o, h1)
        top.addBond(o, h2)
        pos.extend([g + np.array(origin) for g in geo])
    return top, np.array(pos)


def context_for(system, pos, groups=True):
    if groups:
        for i, f in enumerate(system.getForces()):
            f.setForceGroup(i)
    integ = mm.VerletIntegrator(1.0 * unit.femtosecond)
    ctx = mm.Context(system, integ,
                     mm.Platform.getPlatformByName("Reference"))
    ctx.setPositions([Vec3(*p) for p in pos] * unit.nanometer)
    return ctx


def decompose(xml, top, pos, rigid):
    ff = app.ForceField(xml)
    system = ff.createSystem(top, nonbondedMethod=app.NoCutoff,
                             constraints=None, rigidWater=rigid)
    ctx = context_for(system, pos)
    print(f"\n--- force group decomposition (rigidWater={rigid}) ---")
    total = 0.0
    for i, f in enumerate(system.getForces()):
        e = ctx.getState(getEnergy=True, groups={i}).getPotentialEnergy()
        e = e.value_in_unit(unit.kilojoule_per_mole)
        total += e
        print(f"  {f.__class__.__name__:28s} {e:12.5f} kJ/mol")
    print(f"  {'TOTAL':28s} {total:12.5f} kJ/mol")
    print(f"  constraints in system: {system.getNumConstraints()}")
    return system


def find_lj(system):
    """LJ may live in NonbondedForce or in a CustomNonbondedForce."""
    print("\n--- where are the LJ parameters? ---")
    for f in system.getForces():
        if isinstance(f, mm.NonbondedForce):
            print("  NonbondedForce:")
            for i in range(3):
                q, s, e = f.getParticleParameters(i)
                print(f"    atom {i}: q={q.value_in_unit(unit.elementary_charge):+.5f}"
                      f"  sigma={s.value_in_unit(unit.nanometer):.6f}"
                      f"  eps={e.value_in_unit(unit.kilojoule_per_mole):.6f}")
        if isinstance(f, mm.CustomNonbondedForce):
            print(f"  CustomNonbondedForce: {f.getEnergyFunction()[:70]}")
            names = [f.getPerParticleParameterName(k)
                     for k in range(f.getNumPerParticleParameters())]
            print(f"    per-particle params: {names}")
            for i in range(3):
                print(f"    atom {i}: {f.getParticleParameters(i)}")


def scan_oo(xml):
    """Two isolated oxygens: isolates the LJ curve from everything else."""
    top = app.Topology()
    ch = top.addChain()
    for _ in range(2):
        r = top.addResidue("HOH", ch)
        top.addAtom("O", app.Element.getBySymbol("O"), r)
        top.addAtom("H1", app.Element.getBySymbol("H"), r)
        top.addAtom("H2", app.Element.getBySymbol("H"), r)
    print("\n--- O-O interaction vs distance (charges included) ---")
    print("    r(nm)     OpenMM      LJ_ref+Coul_ref    delta")
    ff = app.ForceField(xml)
    a = np.radians(REF["angle"]) / 2.0
    geo = [np.array([0., 0., 0.]),
           np.array([REF["r_OH"] * np.sin(a), REF["r_OH"] * np.cos(a), 0.]),
           np.array([-REF["r_OH"] * np.sin(a), REF["r_OH"] * np.cos(a), 0.])]
    top2, _ = build()
    system = ff.createSystem(top2, nonbondedMethod=app.NoCutoff,
                             constraints=None, rigidWater=True)
    for d in (0.30, 0.35, 0.40, 0.50, 0.70):
        pos = np.array([g for g in geo] +
                       [g + np.array([d, 0, 0]) for g in geo])
        ctx = context_for(system, pos, groups=False)
        e_mm = ctx.getState(getEnergy=True).getPotentialEnergy().value_in_unit(
            unit.kilojoule_per_mole)
        q = [REF["q_O"], REF["q_H"], REF["q_H"]] * 2
        e_ref = 0.0
        for i in range(3):
            for j in range(3, 6):
                r = np.linalg.norm(pos[i] - pos[j])
                e_ref += ONE_4PI_EPS0 * q[i] * q[j] / r
        sr6 = (REF["sigma"] / d) ** 6
        e_ref += 4 * REF["epsilon"] * (sr6 ** 2 - sr6)
        print(f"  {d:7.3f} {e_mm:11.4f} {e_ref:15.4f} {e_mm - e_ref:11.4f}")

xml = "spce.xml"
top, pos = build()
system = decompose(xml, top, pos, rigid=False)
decompose(xml, top, pos, rigid=True)
find_lj(system)
scan_oo(xml)
print("\nIf rigidWater=True kills a large chunk, the gap was intramolecular")
print("strain: your template geometry is not SPC/E's 1.0 A / 109.47 deg.")


--- force group decomposition (rigidWater=False) ---
  HarmonicBondForce                 0.00000 kJ/mol
  NonbondedForce                    3.44700 kJ/mol
  CMMotionRemover                   0.00000 kJ/mol
  HarmonicAngleForce                0.00000 kJ/mol
  TOTAL                             3.44700 kJ/mol
  constraints in system: 0

--- force group decomposition (rigidWater=True) ---
  HarmonicBondForce                 0.00000 kJ/mol
  NonbondedForce                    3.44700 kJ/mol
  CMMotionRemover                   0.00000 kJ/mol
  HarmonicAngleForce                0.00000 kJ/mol
  TOTAL                             3.44700 kJ/mol
  constraints in system: 6

--- where are the LJ parameters? ---
  NonbondedForce:
    atom 0: q=-0.84760  sigma=0.316572  eps=0.649775
    atom 1: q=+0.42380  sigma=1.000000  eps=0.000000
    atom 2: q=+0.42380  sigma=1.000000  eps=0.000000

--- O-O interaction vs distance (charges included) ---
    r(nm)     OpenMM      LJ_ref+Coul_ref    delta
    0.3

In [8]:
ff = app.ForceField(xml)
for i, s in enumerate(ff._scripts):
    print(f"--- script {i}: {len(s.splitlines())} lines ---")
    print(s[:400])
s = ff._scripts[i]          # the long one
print("\n".join(f"{n+1}: {l}" for n, l in
      enumerate(s.splitlines()[1390:1405], start=1390)))

IndexError: list index out of range

In [9]:
ff = app.ForceField("spce.xml")
for name in ["HOH", "TIP3", "TP3M"]:
    t = ff._templates[name]
    print(f"\n{name}")
    for a in t.atoms:
        p = ff._forces  # charges live on the NonbondedForce generator
        print(f"  {a.name:5s} type={a.type}")
    # charges and LJ by type:
    for gen in ff.getGenerators():
        cls = gen.__class__.__name__
        table = getattr(gen, "ljTypes", None) or getattr(gen, "params", None)
        if table is None:
            continue
        for a in t.atoms:
            if a.type in table.paramsForType:
                print(f"   {cls}: {a.name} -> {table.paramsForType[a.type]}")
        for pair, vals in getattr(gen, "nbfixTypes", {}).items():
            if any("HOH" in t or "TIP3" in t or "TP3M" in t for t in pair):
                print("water NBFIX:", pair, vals)
for name in ("TIP3", "TP3M", "HOH"):
    t = ff._templates[name]
    print(name, [(a.name, a.parameters) for a in t.atoms])


HOH
  O     type=spce-O
  H1    type=spce-H
  H2    type=spce-H
   NonbondedGenerator: O -> {'charge': -0.8476, 'sigma': 0.3165719505039882, 'epsilon': 0.6497752}
   NonbondedGenerator: H1 -> {'charge': 0.4238, 'sigma': 1.0, 'epsilon': 0.0}
   NonbondedGenerator: H2 -> {'charge': 0.4238, 'sigma': 1.0, 'epsilon': 0.0}


KeyError: 'TIP3'

In [4]:
ff = app.ForceField('../mosic.xml', 'charmm36_2024/spce.xml')

top, pos = build()
system = ff.createSystem(top, nonbondedMethod=app.NoCutoff,
                         constraints=None, rigidWater=False)


KeyError: 0

In [39]:
"""
Verify that a custom CHARMM-format XML composes correctly with the
CHARMM-mechanism SPC/E water file.

Checks, in order:
  1. template name collisions between the two XMLs
  2. the water file really is SPC/E (charges, LJ, geometry)
  3. every atom is covered by the same LJ force (particle counts)
  4. peptoid <-> water dispersion actually exists and varies with distance

Run:
    python check_composition.py ../mosic.xml charmm36_2024/spce.xml \
                                ../structure_maker/all_res.pdb
"""

import sys
import numpy as np
from openmm import app, unit, Vec3
import openmm as mm

SPCE = dict(q_O=-0.8476, q_H=0.4238, sigma=0.316557, eps=0.650194,
            r_OH=0.1, angle_deg=109.47)
ONE_4PI_EPS0 = 138.935456


# ----------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------
def add_bonds_from_forcefield(topology, forcefield):
    """Add missing bonds to a topology using force field templates."""
    
    # Track existing bonds to avoid duplicates
    existing = set()
    for bond in topology.bonds():
        existing.add((bond[0].index, bond[1].index))
        existing.add((bond[1].index, bond[0].index))
    
    def safe_add(a1, a2):
        if (a1.index, a2.index) not in existing:
            topology.addBond(a1, a2)
            existing.add((a1.index, a2.index))
            existing.add((a2.index, a1.index))
    
    # Intra-residue bonds from templates
    for res in topology.residues():
        if res.name not in forcefield._templates:
            continue
        template = forcefield._templates[res.name]
        atom_map = {a.name: a for a in res.atoms()}
        template_atoms = list(template.atoms)
        for bond in template.bonds:
            a1_name = template_atoms[bond[0]].name
            a2_name = template_atoms[bond[1]].name
            if a1_name in atom_map and a2_name in atom_map:
                safe_add(atom_map[a1_name], atom_map[a2_name])

def water_templates(ff):
    """(name, [atom names]) for every 3-atom, 1-oxygen template."""
    out = []
    for name, t in ff._templates.items():
        el = [a.element.symbol if a.element else "?" for a in t.atoms]
        if len(el) == 3 and el.count("O") == 1 and el.count("H") == 2:
            atoms = [a.name for a in t.atoms]
            o = [a.name for a in t.atoms if a.element.symbol == "O"][0]
            h = [a.name for a in t.atoms if a.element.symbol == "H"]
            out.append((name, o, h))
    return out


def water_geometry(origin):
    a = np.radians(SPCE["angle_deg"]) / 2.0
    r = SPCE["r_OH"]
    return np.array([[0.0, 0.0, 0.0],
                     [r * np.sin(a), r * np.cos(a), 0.0],
                     [-r * np.sin(a), r * np.cos(a), 0.0]]) + np.asarray(origin)


def group_energies(ctx, system):
    out = {}
    for i, f in enumerate(system.getForces()):
        e = ctx.getState(getEnergy=True, groups={i}).getPotentialEnergy()
        out[f.__class__.__name__] = e.value_in_unit(unit.kilojoule_per_mole)
    return out


def make_ctx(system, pos):
    for i, f in enumerate(system.getForces()):
        f.setForceGroup(i)
    ctx = mm.Context(system, mm.VerletIntegrator(1.0 * unit.femtosecond),
                     mm.Platform.getPlatformByName("Reference"))
    ctx.setPositions([Vec3(*p) for p in pos] * unit.nanometer)
    return ctx

def residue_map(ff, topology, water_res=None, water_name="SPCE"):
    """Map each residue to the template of the same name; water explicitly."""
    templates = ff._templates
    m = {}
    for res in topology.residues():
        if water_res is not None and res is water_res:
            m[res] = water_name
        elif res.name in templates:
            m[res] = res.name
        else:
            print(f"   note: no template named {res.name!r}, leaving to matcher")
    return m

# ----------------------------------------------------------------------
# 1. template collisions
# ----------------------------------------------------------------------
def check_collisions(custom_xml, water_xml):
    print("=" * 62)
    print("1. template collisions")
    a = set(app.ForceField(custom_xml)._templates)
    b = set(app.ForceField(water_xml)._templates)
    both = sorted(a & b)
    if both:
        print(f"   OVERLAP ({len(both)}): {both}")
        print("   -> remove these <Residue> blocks from the custom XML")
        print("      (keep <Type> and <LennardJonesForce> entries)")
    else:
        print("   none  [PASS]")
    return both


# ----------------------------------------------------------------------
# 2. is the water file actually SPC/E?
# ----------------------------------------------------------------------
def check_water(water_xml):
    print("=" * 62)
    print("2. water model identity")
    ff = app.ForceField(water_xml)
    tmpls = water_templates(ff)
    print(f"   water-like templates: {[t[0] for t in tmpls]}")
    name, o_name, h_names = tmpls[0]

    top = app.Topology()
    residues = []
    for shift in ([0, 0, 0], [0.4684, 0, 0]):
        ch = top.addChain()
        r = top.addResidue(name, ch)
        residues.append(r)
        o = top.addAtom(o_name, app.Element.getBySymbol("O"), r)
        for hn in h_names:
            h = top.addAtom(hn, app.Element.getBySymbol("H"), r)
            top.addBond(o, h)
    pos = np.vstack([water_geometry([0, 0, 0]),
                     water_geometry([0.4684, 0, 0])])

    system = ff.createSystem(top, nonbondedMethod=app.NoCutoff,
                             constraints=None, rigidWater=False,
                             residueTemplates={r: name for r in residues})
    ctx = make_ctx(system, pos)

    nb = next(f for f in system.getForces() if isinstance(f, mm.NonbondedForce))
    q_o = nb.getParticleParameters(0)[0].value_in_unit(unit.elementary_charge)
    q_h = nb.getParticleParameters(1)[0].value_in_unit(unit.elementary_charge)
    print(f"   q_O = {q_o:+.5f}  (SPC/E {SPCE['q_O']:+.5f})"
          f"   {'PASS' if abs(q_o - SPCE['q_O']) < 1e-4 else 'CHECK'}")
    print(f"   q_H = {q_h:+.5f}  (SPC/E {SPCE['q_H']:+.5f})")

    for f in system.getForces():
        if isinstance(f, mm.HarmonicBondForce) and f.getNumBonds():
            r0 = f.getBondParameters(0)[2].value_in_unit(unit.nanometer)
            print(f"   r_OH = {r0:.5f} nm  (SPC/E 0.10000)"
                  f"   {'PASS' if abs(r0 - 0.1) < 1e-4 else 'CHECK'}")
        if isinstance(f, mm.HarmonicAngleForce) and f.getNumAngles():
            ang = f.getAngleParameters(0)[3].value_in_unit(unit.degrees)
            print(f"   angle = {ang:.3f} deg  (SPC/E 109.470)"
                  f"   {'PASS' if abs(ang - 109.47) < 0.05 else 'CHECK'}")

    # LJ: whichever force carries it, compare to the SPC/E O-O reference
    sr6 = (SPCE["sigma"] / 0.4684) ** 6
    ref = 4 * SPCE["eps"] * (sr6 ** 2 - sr6)
    e = group_energies(ctx, system)
    lj = e.get("CustomNonbondedForce", 0.0)
    if lj == 0.0:
        # LJ is inside NonbondedForce; back it out from the Coulomb sum
        q = [SPCE["q_O"], SPCE["q_H"], SPCE["q_H"]] * 2
        coul = sum(ONE_4PI_EPS0 * q[i] * q[j] /
                   np.linalg.norm(pos[i] - pos[j])
                   for i in range(3) for j in range(3, 6))
        lj = e.get("NonbondedForce", 0.0) - coul
    print(f"   LJ(O-O) at 0.4684 nm = {lj:.4f} kJ/mol  (SPC/E {ref:.4f})")
    print(f"   {'PASS' if abs(lj - ref) < 2e-3 else 'CHECK - eps_H nonzero?'}")


# ----------------------------------------------------------------------
# 3 + 4. composition with the peptoid
# ----------------------------------------------------------------------
def check_cross_terms(custom_xml, water_xml, pdb_path):
    print("=" * 62)
    print("3. LJ coverage and 4. peptoid-water dispersion")
    ff = app.ForceField(custom_xml, water_xml)
    pdb = app.PDBFile(pdb_path)
    add_bonds_from_forcefield(pdb.topology, ff)

    # take the water template from the WATER file, not the merged pool
    wname, o_name, h_names = water_templates(app.ForceField(water_xml))[0]

    top = app.Topology()
    # copy the peptoid
    amap = {}
    for ch in pdb.topology.chains():
        nch = top.addChain()
        for res in ch.residues():
            nres = top.addResidue(res.name, nch)
            for at in res.atoms():
                amap[at] = top.addAtom(at.name, at.element, nres)
    for b in pdb.topology.bonds():
        top.addBond(amap[b[0]], amap[b[1]])

    n_pep = top.getNumAtoms()
    pep_pos = np.array(pdb.positions.value_in_unit(unit.nanometer))

    # pick the most exposed heavy atom so the probe water sees mainly it
    heavy = [a.index for a in top.atoms() if a.element.symbol != "H"]
    centroid = pep_pos.mean(axis=0)
    d = np.linalg.norm(pep_pos[heavy] - centroid, axis=1)
    target = heavy[int(np.argmax(d))]
    outward = (pep_pos[target] - centroid)
    outward /= np.linalg.norm(outward)
    tgt_atom = list(top.atoms())[target]
    print(f"   probe atom: {tgt_atom.residue.name}/{tgt_atom.name} "
          f"(index {target}, element {tgt_atom.element.symbol})")

    # append one water
    wch = top.addChain()
    wres = top.addResidue(wname, wch)
    wo = top.addAtom(o_name, app.Element.getBySymbol("O"), wres)
    for hn in h_names:
        wh = top.addAtom(hn, app.Element.getBySymbol("H"), wres)
        top.addBond(wo, wh)

    try:
        rmap = residue_map(ff, top, water_res=wres, water_name=wname)
        system = ff.createSystem(top, nonbondedMethod=app.NoCutoff,
                             constraints=None, rigidWater=True,
                             residueTemplates=rmap)
    except Exception as exc:
        print(f"   createSystem FAILED: {exc}")
        return

    # --- check 3: does one LJ force cover every atom?
    print()
    for f in system.getForces():
        if isinstance(f, mm.CustomNonbondedForce):
            n = f.getNumParticles()
            ok = "PASS" if n == system.getNumParticles() else "CHECK"
            print(f"   CustomNonbondedForce covers {n} of "
                  f"{system.getNumParticles()} particles   [{ok}]")

    # --- check 4: scan the water away from the probe atom
    print()
    print("     r(nm)   CustomNonbonded   NonbondedForce")
    ref_e = None
    rows = []
    for r in (0.30, 0.35, 0.40, 0.50, 0.80, 5.00):
        pos = np.vstack([pep_pos,
                         water_geometry(pep_pos[target] + outward * r)])
        ctx = make_ctx(system, pos)
        e = group_energies(ctx, system)
        rows.append((r, e.get("CustomNonbondedForce", 0.0),
                     e.get("NonbondedForce", 0.0)))
        print(f"   {r:6.2f} {rows[-1][1]:16.4f} {rows[-1][2]:16.4f}")

    far = rows[-1][1]
    contact = rows[2][1]          # r = 0.40 nm
    close = rows[0][1]            # r = 0.30 nm
    print()
    print(f"   cross-term LJ at 0.40 nm: {contact - far:+.4f} kJ/mol")
    print(f"   cross-term LJ at 0.30 nm: {close - far:+.4f} kJ/mol")
    if abs(contact - far) < 1e-6:
        print("   [FAIL] no distance dependence - peptoid and water are in")
        print("          separate LJ forces with no cross-term.")
    elif (close - far) > (contact - far):
        print("   [PASS] attractive near contact, repulsive closer in.")
    else:
        print("   [CHECK] varies with distance but not in the expected shape.")

custom, water, pdb_path = "../mosic.xml", "charmm36_2024/spce.xml", "../structure_maker/all_res.pdb"
check_collisions(custom, water)
check_water(water)
check_cross_terms(custom, water, pdb_path)

1. template collisions
   none  [PASS]
2. water model identity
   water-like templates: ['SPCE']
   q_O = -0.84760  (SPC/E -0.84760)   PASS
   q_H = +0.42380  (SPC/E +0.42380)
   r_OH = 0.10000 nm  (SPC/E 0.10000)   PASS
   angle = 109.470 deg  (SPC/E 109.470)   PASS
   LJ(O-O) at 0.4684 nm = -0.2241 kJ/mol  (SPC/E -0.2242)
   PASS
3. LJ coverage and 4. peptoid-water dispersion
   probe atom: ACEp/CAY (index 1, element C)

   CustomNonbondedForce covers 712 of 712 particles   [PASS]

     r(nm)   CustomNonbonded   NonbondedForce
     0.30    22598302.3509        -178.3065
     0.35    22598280.0052        -176.4870
     0.40    22598279.3427        -175.4577
     0.50    22598280.1061        -174.4464
     0.80    22598280.5309        -173.7772
     5.00    22598280.5724        -173.6885

   cross-term LJ at 0.40 nm: -1.2298 kJ/mol
   cross-term LJ at 0.30 nm: +21.7785 kJ/mol
   [PASS] attractive near contact, repulsive closer in.


# Phase 1: Reproduce Published SPC/E Numbers

In [7]:
import os, copy, tempfile
import numpy as np
import mdtraj as md
from openmm import app, unit, Vec3
import openmm as mm
from openmmtools import alchemy, states, mcmc, multistate

C4_CONV     = 4.184 * 1e-4      # kcal*A^4/mol -> kJ*nm^4/mol
ALPHA_WATER = 1.444             # A^3
XI_EW       = 2.837297
ONE_4PI_EPS0 = 138.935456       # kJ*nm/(mol*e^2)
KCAL        = 4.184
TEMP      = 298.15 * unit.kelvin
PRES      = 1.0 * unit.bar
CUTOFF    = 1.0 * unit.nanometer
PME_ALPHA = 3.12 / unit.nanometer
DT        = 2.0 * unit.femtosecond

CFG = dict(
    name="NA", element="Na", mass=22.99, charge=1.0,
    sigma=0.2439,      # nm
    epsilon=0.3658,    # kJ/mol
    c4=6,           # kcal*A^4/mol; None or 0 -> plain 12-6
    box=3.0,           # nm
)

In [33]:
ION_XML = """<ForceField>
 <AtomTypes>
  <Type name="ION-{name}" class="{name}" element="{element}" mass="{mass}"/>
 </AtomTypes>
 <Residues>
  <Residue name="{name}"><Atom name="{name}" type="ION-{name}"/></Residue>
 </Residues>
 <NonbondedForce coulomb14scale="1.0" lj14scale="1.0">
  <Atom type="ION-{name}" charge="{charge}" sigma="{sigma}" epsilon="{epsilon}"/>
 </NonbondedForce>
</ForceField>
"""

def build(cfg):
    fd, path = tempfile.mkstemp(suffix=".xml")
    with os.fdopen(fd, "w") as fh:
        fh.write(ION_XML.format(**cfg))
    ff = app.ForceField("charmm36/spce.xml", path)

    top = app.Topology()
    res = top.addResidue(cfg["name"], top.addChain())
    top.addAtom(cfg["name"], app.Element.getBySymbol(cfg["element"]), res)

    modeller = app.Modeller(top, [Vec3(0, 0, 0)] * unit.nanometer)
    modeller.addSolvent(ff, model="spce",
                        boxSize=Vec3(1, 1, 1) * cfg["box"] * unit.nanometer,
                        neutralize=False)
    print(f"{modeller.topology.getNumResidues()-1} waters, ion index 0")
    return ff, modeller, path

ff, modeller, ion_xml_path = build(CFG)

Exception: Multiple non-identical matching templates found for residue 1 (NA): SOD, NA.

In [ ]:
def make_system(ff, topology, cfg):
    system = ff.createSystem(topology, nonbondedMethod=app.PME,
                             nonbondedCutoff=CUTOFF, constraints=None,
                             rigidWater=True)
    nb = next(f for f in system.getForces() if isinstance(f, mm.NonbondedForce))
    g = int(np.ceil(cfg["box"] / 0.08)); g += g % 2
    nb.setPMEParameters(PME_ALPHA, g, g, g)
    nb.setUseDispersionCorrection(False)   # DispCorr = No
    nb.setUseSwitchingFunction(False)
    return system

def add_c4(system, topology, c4_kcal, switch_frac=0.9):
    """Adds -lambda_c4 * C4/r^4 for ion <-> water only."""
    nb = next(f for f in system.getForces() if isinstance(f, mm.NonbondedForce))
    cnb = mm.CustomNonbondedForce(
        "-lambda_c4*(cfour1*arel2 + cfour2*arel1)/(r^4)")
    cnb.addGlobalParameter("lambda_c4", 1.0)
    cnb.addPerParticleParameter("cfour")
    cnb.addPerParticleParameter("arel")

    ions, others = [], []
    for atom in topology.atoms():
        if atom.index == 0:
            cnb.addParticle([c4_kcal * C4_CONV, 0.0]); ions.append(0)
        elif atom.element.symbol == "O":
            cnb.addParticle([0.0, 1.0]); others.append(atom.index)
        else:
            cnb.addParticle([0.0, 0.0]); others.append(atom.index)

    cnb.addInteractionGroup(ions, others)
    cnb.setNonbondedMethod(mm.CustomNonbondedForce.CutoffPeriodic)
    cnb.setCutoffDistance(nb.getCutoffDistance())
    cnb.setUseSwitchingFunction(True)
    cnb.setSwitchingDistance(nb.getCutoffDistance() * switch_frac)
    cnb.setUseLongRangeCorrection(False)   # corrected analytically instead
    for k in range(nb.getNumExceptions()):
        p1, p2, *_ = nb.getExceptionParameters(k)
        cnb.addExclusion(p1, p2)
    system.addForce(cnb)
    return cnb

class C4State(states.GlobalParameterState):
    lambda_c4 = states.GlobalParameterState.GlobalParameter(
        "lambda_c4", standard_value=1.0)

USE_C4 = bool(CFG["c4"])

In [12]:
system = make_system(ff, modeller.topology, CFG)
if USE_C4:
        add_c4(system, modeller.topology, CFG["c4"])


In [ ]:
def equilibrate(ff, modeller, cfg, ns=0.5):
    system = make_system(ff, modeller.topology, cfg)
    if USE_C4:
        add_c4(system, modeller.topology, cfg["c4"])
    system.addForce(mm.MonteCarloBarostat(PRES, TEMP, 25))
    integ = mm.LangevinMiddleIntegrator(TEMP, 1.0/unit.picosecond, DT)
    sim = app.Simulation(modeller.topology, system, integ)
    sim.context.setPositions(modeller.positions)
    sim.minimizeEnergy()
    sim.context.setVelocitiesToTemperature(TEMP)
    sim.step(int(ns * unit.nanoseconds / DT))
    st = sim.context.getState(getPositions=True, enforcePeriodicBox=True)
    return st.getPositions(), st.getPeriodicBoxVectors()

eq_positions, eq_box = equilibrate(ff, modeller, CFG)
print("equilibrated box:", eq_box[0][0])

KeyboardInterrupt: 

# Phase 2: Imidazole and Acetate Moieties

In this phase, we will adapt the parameters from the original Merz parameterization paper to work correctly within our new MoSiC forcefield, 


In [32]:
CFG = dict(
    name="NA", element="Na", mass=22.99, charge=1.0,
    sigma=0.2439,      # nm
    epsilon=0.3658,    # kJ/mol
    c4=6,           # kcal*A^4/mol; None or 0 -> plain 12-6
    box=3.0,           # nm
)
ION_XML = """<ForceField>
 <AtomTypes>
  <Type name="ION-{name}" class="{name}" element="{element}" mass="{mass}"/>
 </AtomTypes>
 <Residues>
  <Residue name="{name}"><Atom name="{name}" type="ION-{name}"/></Residue>
 </Residues>
 <NonbondedForce coulomb14scale="0.833333" lj14scale="0.5">
  <Atom type="ION-{name}" charge="{charge}" sigma="{sigma}" epsilon="{epsilon}"/>
 </NonbondedForce>
</ForceField>
"""

def build(cfg):
    fd, path = tempfile.mkstemp(suffix=".xml")
    with os.fdopen(fd, "w") as fh:
        fh.write(ION_XML.format(**cfg))
    ff = app.ForceField("../spce_ions.xml", path)

    top = app.Topology()
    res = top.addResidue(cfg["name"], top.addChain())
    top.addAtom(cfg["name"], app.Element.getBySymbol(cfg["element"]), res)

    modeller = app.Modeller(top, [Vec3(0, 0, 0)] * unit.nanometer)
    modeller.addSolvent(ff, model="spce",
                        boxSize=Vec3(1, 1, 1) * cfg["box"] * unit.nanometer,
                        neutralize=False)
    print(f"{modeller.topology.getNumResidues()-1} waters, ion index 0")
    return ff, modeller, path

ff, modeller, ion_xml_path = build(CFG)
list(modeller.topology.atoms())[1].name

ValueError: Found multiple NonbondedForce tags with different 1-4 scales

In [3]:
WATER_POL = 1.444
DEFAULT_C4_PARAMS = { 
    'SPCE' : {
        'Li1': 33.0, 'Na1': 6.0, 'K1': 19.0, 'Rb1': 7.0, 'Cs1': 12.0,
        'Tl1': 61.0, 'Cu1': 9.0, 'Ag1': 92.0, 'F-1': -53.0,
        'Cl-1': -55.0, 'Br-1': -51.0, 'I-1': -51.0, 'Be2': 188.1,
        'Cu2': 304.4, 'Ni2': 205.2, 'Zn2': 231.2, 'Co2': 209.2,
        'Cr2': 131.2, 'Fe2': 155.4, 'Mg2': 122.2, 'V2': 206.6,
        'Mn2': 154.9, 'Hg2': 300.2, 'Cd2': 198.8, 'Ca2': 89.0,
        'Sn2': 201.1, 'Sr2': 96.3, 'Ba2': 85.8, 'Al3': 406.0,
        'Fe3': 442.0, 'Cr3': 254.0, 'In3': 349.0, 'Tl3': 455.0,
        'Y3': 209.0, 'La3': 165.0, 'Ce3': 242.0, 'Pr3': 272.0,
        'Nd3': 235.0, 'Sm3': 224.0, 'Eu3': 273.0, 'Gd3': 186.0,
        'Tb3': 227.0, 'Dy3': 206.0, 'Er3': 247.0, 'Tm3': 262.0,
        'Lu3': 247.0, 'Hf4': 810.0, 'Zr4': 760.0, 'Ce4': 694.0,
        'U4': 1043.0, 'Pu4': 828.0, 'Th4': 513.0
    }
}

In [ ]:
import warnings
# Reformatting/fixing ParmEd's add1264.py to work for our system within OpenMM
def params1264(top, met_ind, c4file, watermodel, polfile):
    try:
        pollist = _get_params(polfile)
    except ValueError as err:
        raise ValueError(
            f'Bad polarizability file {polfile}. Expected a file with '
            '2 columns: <Amber Atom Type> <Polarizability>'
        ) from err

    if c4file is None:
        c4list = DEFAULT_C4_PARAMS[watermodel]
    else:
        try:
            c4list = _get_params(c4file)
        except ValueError as err:
            raise ValueError(
                f'Bad C4 parameter file {c4file}. Expected a file with 2 columns: '
                '<Atom Element> <C4 Parameter>'
            ) from err

    # Determine metal type
    
def _get_params(fname):
    params = dict()

    with open(fname, 'r') as f:
        for line in f:
            atomtype, param = line.split()[:2]
            param = float(param)
            if atomtype in params and abs(param - params[atomtype]) > 0.0001:
                warnings.warn(f'Atom type {atomtype} has multiple parameters in {fname}.',
                              UserWarning)
            params[atomtype] = param

        return params